In [48]:
import requests
import pandas as pd
import certifi
import ssl
import geopy.geocoders

ctx = ssl.create_default_context(cafile=certifi.where())
geopy.geocoders.options.default_ssl_context = ctx

In [11]:
# Paste your credentials from https://www.strava.com/settings/api
CLIENT_ID = "230098"
CLIENT_SECRET = "57ff9f08daffc8d77771e03bae7c380eed00d7b0"
REFRESH_TOKEN = "5d06f94ff6551fe71a0c8ec609a90b152f0997f9"
CODE = "3f3ba03eb1e194890525598ba498e8cdb5054e55"

In [ ]:
# One-time step: exchange the authorization code for a properly scoped refresh token.
# After running this cell once, copy the new refresh_token into the cell above and skip this cell.
# auth_response = requests.post(
#     "https://www.strava.com/oauth/token",
#     data={
#         "client_id": CLIENT_ID,
#         "client_secret": CLIENT_SECRET,
#         "code": CODE,
#         "grant_type": "authorization_code",
#     },
# )
# new_tokens = auth_response.json()
# print(new_tokens)

{'token_type': 'Bearer', 'expires_at': 1777100443, 'expires_in': 21600, 'refresh_token': '5d06f94ff6551fe71a0c8ec609a90b152f0997f9', 'access_token': 'e701fa279ba145db29ac43fe5d98a0e85483901d', 'scope': 'activity:read_all read', 'athlete': {'id': 65590982, 'username': 'wynware_athlete', 'resource_state': 2, 'firstname': 'Wynne', 'lastname': 'Philpott', 'bio': None, 'city': None, 'state': None, 'country': None, 'sex': 'M', 'premium': False, 'summit': False, 'created_at': '2020-08-06T00:10:14Z', 'updated_at': '2024-04-16T01:39:16Z', 'badge_type_id': 0, 'profile_medium': 'https://lh3.googleusercontent.com/a/ACg8ocKmLVJ3A0pZXqLCWIr6sST54Mp1J-QeyK5fYmfT7bEjusZmM6s=s96-c', 'profile': 'https://lh3.googleusercontent.com/a/ACg8ocKmLVJ3A0pZXqLCWIr6sST54Mp1J-QeyK5fYmfT7bEjusZmM6s=s96-c', 'friend': None, 'follower': None}}


In [12]:
# Access tokens expire every 6 hours — use the refresh token to get a fresh one
token_response = requests.post(
    "https://www.strava.com/oauth/token",
    data={
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "refresh_token": REFRESH_TOKEN,
        "grant_type": "refresh_token",
    },
)
tokens = token_response.json()
access_token = tokens["access_token"]
print("Token refreshed successfully")

Token refreshed successfully


In [13]:
headers = {"Authorization": f"Bearer {access_token}"}

# Verify connection by fetching your athlete profile
athlete = requests.get("https://www.strava.com/api/v3/athlete", headers=headers).json()
print(f"{athlete['firstname']} {athlete['lastname']}")

Wynne Philpott


In [14]:
# Fetch your activities (paginated, 200 max per page)
activities = requests.get(
    "https://www.strava.com/api/v3/athlete/activities",
    headers=headers,
    params={"per_page": 200, "page": 1},
).json()

df = pd.json_normalize(activities)
print(f"{len(df)} activities loaded")
df.head()

200 activities loaded


,resource_state,name,distance,moving_time,elapsed_time,total_elevation_gain,type,sport_type,workout_type,device_name,...,pr_count,total_photo_count,has_kudoed,athlete.id,athlete.resource_state,map.id,map.summary_polyline,map.resource_state,average_cadence,average_temp
0,2,Afternoon Run,6554.2,1920,1920,6.4,Run,Run,NaN,Strava App,...,3,0,False,65590982,1,a18204409499,kz{fEhvnqQQ{ATSR_@^iAPo@NaABy@BGBC?IDWAS@EEqAA...,2,NaN,NaN
1,2,Afternoon Run,16162.9,5506,5506,33.2,Run,Run,NaN,Strava App,...,1,0,False,65590982,1,a18178023435,s|{fE`unqQhA}@?Gd@_An@gCF_ADMDeBGkFEiAEO?UGc@W...,2,NaN,NaN
2,2,Afternoon Run,11330.2,3661,3661,17.0,Run,Run,NaN,Strava App,...,1,0,False,65590982,1,a18125013321,c}{fEfunqQxAmAV_@Pg@Tq@AIRi@J}@?a@Ha@B_AEMBk@G...,2,NaN,NaN
3,2,Afternoon Run,6512.1,2286,2304,56.7,Run,Run,NaN,Strava App,...,0,0,False,65590982,1,a18096493802,wk|fEzbhqQEACBCr@@XCn@@RET?j@Dj@Et@CDD`@S|A?VE...,2,NaN,NaN
4,2,Afternoon Run,6455.7,2348,2353,53.9,Run,Run,NaN,Strava App,...,0,0,False,65590982,1,a18019486053,ck|fEpbhqQa@nAARBJANBVAp@@HAl@EN@\Ff@K\Ij@Ad@@...,2,NaN,NaN


In [16]:
segments = requests.get(
    "https://www.strava.com/api/v3/segments/explore",
    headers=headers,
    params={
        "bounds": "30.2,-97.8,30.3,-97.7",  # sw_lat, sw_lng, ne_lat, ne_lng
        "activity_type": "running",
    },
).json()

segments_df = pd.json_normalize(segments["segments"])
segments_df.head()

,id,resource_state,name,climb_category,climb_category_desc,avg_grade,start_latlng,end_latlng,elev_difference,distance,points,starred,elevation_profile,local_legend_enabled
0,1834651,2,E Riverside Dr Climb,1,4,23.6,"[30.255058519542217, -97.73969281464815]","[30.252749221399426, -97.73740447126329]",82.8,350.4,aetwDbxpsQPE^MNAf@OZYj@Sd@g@t@g@b@]h@q@Rc@`@c@...,False,https://d3o5xota0a1fcr.cloudfront.net/v6/chart...,True
1,2798633,2,Turkey Trot 2012,0,NC,-0.1,"[30.2637894, -97.7475571]","[30.2614102, -97.7505802]",46.9,8164.6,s{uwDfirsQQMMOq@mAMQYSKOE?EK_@OMWAEEGe@Fm@AQOE...,False,https://d3o5xota0a1fcr.cloudfront.net/v6/chart...,True
2,3101565,2,5 mile loop Mopac / Congress Ave,0,NC,0.0,"[30.2749058, -97.7707132]","[30.2749339, -97.7707354]",13.0,7682.1,caxwD~yvsQOh@DThBz@^\hA`@~@j@XLZf@ZZREZSNSRKxA...,False,https://d3o5xota0a1fcr.cloudfront.net/v6/chart...,True
3,2466065,2,The Exposition Mile,0,NC,-1.6,"[30.2965795, -97.7681945]","[30.2831889, -97.7766742]",27.2,1693.7,sh|wDfjvsQDLPLjAl@p@VNL`B~@TPt@^rBrAhA`@h@ZZLP...,False,https://d3o5xota0a1fcr.cloudfront.net/v6/chart...,True
4,3473359,2,Mike's Moonlight / Strava Segment 1,0,NC,0.0,"[30.2653655, -97.7638382]","[30.2649918, -97.7568458]",3.0,760.0,oevwD~nusQSU{@gB_@kAq@}Ai@gBO]Su@O_@Ia@?a@tAyE...,False,https://d3o5xota0a1fcr.cloudfront.net/v6/chart...,True


In [46]:
segments_df[["id", "name"]]

,id,name
0,2849251,Fort Worth Turkey Trot 10K
1,21040666,Full Red Loop (up to exit of forest)
2,5495833,Acme Brick
3,12979694,Bridge Sprint
4,10978336,Clearfork Food Truck Sprint Half Mile
5,9254684,Marine Creek Lake Counterclockwise
6,17097813,Bird Poop Drop Zone
7,26987108,How many times have I run this?
8,12630111,Spookies
9,7410174,Oakmont to I-20 (Bellaire Dr.)


In [45]:
segments = requests.get(
    "https://www.strava.com/api/v3/segments/explore",
    headers=headers,
    params={
        "bounds": "32.65,-97.50,32.85,-97.20",
        "activity_type": "running",
    },
).json()

segments_df = pd.json_normalize(segments["segments"])

In [28]:
segment_detail = requests.get(
    f"https://www.strava.com/api/v3/segments/{segments_df['id'].iloc[0]}",
    headers=headers,
).json()
print(segment_detail.keys())

dict_keys(['id', 'resource_state', 'name', 'activity_type', 'distance', 'average_grade', 'maximum_grade', 'elevation_high', 'elevation_low', 'start_latlng', 'end_latlng', 'elevation_profile', 'elevation_profiles', 'climb_category', 'city', 'state', 'country', 'private', 'hazardous', 'starred', 'created_at', 'updated_at', 'total_elevation_gain', 'map', 'effort_count', 'athlete_count', 'star_count', 'athlete_segment_stats', 'xoms', 'local_legend'])


In [36]:
unique_ids = segments_df["id"].unique()

details = []
for seg_id in segments_df["id"]:
    detail = requests.get(
        f"https://www.strava.com/api/v3/segments/{2849251}",
        headers=headers,
    ).json()
    details.append(detail)

details_df = pd.json_normalize(details)

In [49]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="data-science-app")
location = geolocator.geocode("Fort Worth, TX")
print(location.latitude, location.longitude)  # 32.7554883, -97.3307658

32.753177 -97.3327459


In [50]:
def get_bounds(city, offset=0.1):
    geolocator = Nominatim(user_agent="data-science-app")
    location = geolocator.geocode(city)
    lat, lng = location.latitude, location.longitude
    return f"{lat - offset},{lng - offset},{lat + offset},{lng + offset}"

bounds = get_bounds("Fort Worth, TX")
# "32.655,-97.431,32.855,-97.231"

In [52]:
denver_bounds = get_bounds("Denver, CO")

In [53]:
denver_bounds

'39.6392364,-105.084862,39.839236400000004,-104.88486200000001'

In [57]:
denver_segments = requests.get(
    "https://www.strava.com/api/v3/segments/explore",
    headers=headers,
    params={"bounds": denver_bounds, "activity_type": "running"},
).json()

denver_segments_df = pd.json_normalize(denver_segments["segments"])

In [59]:
denver_segments_df[["id", "name"]]

,id,name
0,1389672,Sloan's Lake - Inner Loop
1,4243033,Wash Park NW Start (counter clockwise)
2,21321765,Brooks Best Fest Denver
3,9582774,CP-Hill Climb
4,23954874,Clarkson Hill
5,8809682,Rocky Mountain Lake Backside
6,24196826,21st to23rd Sprint Challenge
7,2972371,1 mile TT
8,2451250,Cherry Creek Dash
9,5617807,1st ave hudson to park
